<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 8 — Data Analysis Reference

A lookup guide. Notebooks 01 and 02 are meant to be read in order; this one is
meant to be **searched**. Keep it open while you work on Homework 7.

Every section runs on the small table `plants` defined in the first cell, so
you can change any example and see what happens. Run the setup cell, then jump
to the section you need.

| Section | When you need it |
|---|---|
| [A — Inspecting a table](#A) | first, always |
| [B — Types, and converting them](#B) | a number is text, a date is text |
| [C — Missing values](#C) | finding, counting, filling, creating, and how they behave |
| [D — Whole-column operations](#D) | a new column without a loop |
| [E — Text and date columns](#E) | `.str` and `.dt` |
| [F — Split, apply, combine](#F) | anything "per group" |
| [G — Reading and writing CSV and JSON](#G) | files |
| [H — Working style](#H) | copies, chaining, and the warnings you will meet |
| [I — From loop to pandas](#I) | "I know how to do this with a loop..." |

**Not in this notebook** because Session 7 covered them:

| Topic | Where |
|---|---|
| Choosing rows and columns, sorting, `.loc` and `.iloc` | `07-Data-Tables/01_Intro_Numpy_Pandas`, `03_Array_DataFrame_Operations` |
| `merge` and the kinds of join | `07-Data-Tables/02_Northwind_SQL_Pandas`, Part 4 |
| Ranking and running totals within a group | `07-Data-Tables/02_Northwind_SQL_Pandas`, Part 6 |
| `melt`, `pivot`, `resample`, `rolling`, `shift` | `07-Data-Tables/03_Array_DataFrame_Operations`, Part 2 |

In [ ]:
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 160


def make_plants():
    """A fresh copy of the example table.  Call it again to start over."""
    return pd.DataFrame({
        "id":      [1, 2, 3, 4, 5, 6, 7, 8],
        "species": ["thistle", "thistle", "knapweed", "spurge", "cheatgrass",
                    "thistle", "knapweed", "spurge"],
        "forest":  ["Alpha", "Alpha", "Alpha", "Bravo", "Bravo", "Bravo",
                    "Charlie", "Charlie"],
        "lifeform": ["Forb", "Forb", "Unknown", "Forb", "Grass", None,
                     " unknown ", "Forb"],
        "date":    ["2015/06/01", "2015/07/15", "2016/08/20", "1900/01/01",
                    "2016/05/05", "2017/06/30", "", "2017/08/08"],
        "area":    [2.0, 0.5, 10.0, 4.0, 150.0, 1.0, 8.0, 0.0],
        "percent": [50, 100, 25, 75, 10, 100, 50, 100],
        "cover":   [10.0, 30.0, np.nan, 20.0, np.nan, 150.0, np.nan, np.nan],
    })


plants = make_plants()
plants

---
<a id="A"></a>

## A — Inspecting a table

| You want to know | Write | Gives |
|---|---|---|
| how big it is | `df.shape` | `(rows, columns)` |
| what the columns are | `df.columns.tolist()` | a list of names |
| the type of each column | `df.dtypes` | one type per column |
| types, counts and memory together | `df.info()` | a printed summary |
| what rows look like | `df.head(n)`, `df.tail(n)`, `df.sample(n)` | a few rows |
| missing values per column | `df.isna().sum()` | a count per column |
| the share missing per column | `df.isna().mean()` | a fraction per column |
| different values per column | `df.nunique()` | a count per column |
| the numeric columns in summary | `df.describe()` | count, mean, std, min, quartiles, max |
| the text columns in summary | `df.describe(include="object")` | count, unique, most common |
| how often each value occurs | `df["col"].value_counts(dropna=False)` | counts, most common first |
| whether a column is a key | `df["col"].is_unique` | `True` or `False` |
| repeated rows | `df.duplicated().sum()` | a count |

In [ ]:
print(plants.shape)
print()
print(plants.dtypes)

In [ ]:
# One table with the three most useful facts per column.
pd.DataFrame({
    "type": plants.dtypes.astype(str),
    "missing": plants.isna().sum(),
    "share": plants.isna().mean(),
    "distinct": plants.nunique(),
})

In [ ]:
plants.describe()

In [ ]:
# dropna=False shows the missing values as a row of their own.
plants["lifeform"].value_counts(dropna=False)

**Reading `describe()`.** `count` is the number of values **present**, so a
`count` below the number of rows is a column with blanks. Compare `min` and
`max` with what is physically possible: a `cover` of 150 %, an `area` of 0.
Compare `mean` with the `50%` row (the median): far apart means a few extreme
values are pulling the mean.

**`sample(n)` over `head(n)`.** The first rows of a file are often the oldest
or the tidiest. A random handful is more honest. `sample(5, random_state=1)`
gives the same handful every time.

---
<a id="B"></a>

## B — Types, and converting them

| pandas calls it | It holds | Usual cause of trouble |
|---|---|---|
| `int64` | whole numbers | cannot hold a missing value; pandas switches the column to `float64` |
| `float64` | decimals, and whole numbers with blanks | `2015.0` where you expected `2015` |
| `bool` | `True` / `False` | becomes `object` if a blank gets in |
| `object` | text, or anything mixed | a number column with one stray word |
| `datetime64[ns]` | dates and times | arrives as text unless converted |
| `category` | text from a small fixed set | saves memory; see Session 7 notebook 03 |

| Convert | Write | What happens to a value that will not convert |
|---|---|---|
| text → number | `pd.to_numeric(col, errors="coerce")` | becomes missing |
| text → date | `pd.to_datetime(col, errors="coerce")` | becomes missing (`NaT`) |
| text → date, known layout | `pd.to_datetime(col, format="%Y/%m/%d", errors="coerce")` | becomes missing; faster and unambiguous |
| anything → text | `col.astype("string")` or `col.astype(str)` | — |
| decimal → whole number | `col.astype(int)` | **error** if any value is missing |
| decimal → whole number, blanks allowed | `col.astype("Int64")` (capital I) | stays missing |

`errors="coerce"` is the important option. Without it, one bad value stops the
whole conversion with an error. With it, the bad values become missing and you
can count them.

In [ ]:
prices = pd.Series(["12.50", "30", "n/a", "$18.00", ""])

print(pd.to_numeric(prices, errors="coerce").tolist())
print("could not be read:", pd.to_numeric(prices, errors="coerce").isna().sum())

In [ ]:
# Dates.  The empty text and anything that is not a date become NaT.
dates = pd.to_datetime(plants["date"], format="%Y/%m/%d", errors="coerce")
pd.DataFrame({"text": plants["date"], "date": dates, "year": dates.dt.year})

The `year` column shows `2015.00`, not `2015`: one row has no date, a column
with a blank cannot be `int64`, so pandas made it `float64`. Two ways out:

In [ ]:
print(dates.dt.year.astype("Int64").tolist())          # whole numbers that allow a blank
print(dates.dropna().dt.year.astype(int).tolist())     # or drop the blanks first

**Format codes** for `format=`: `%Y` four-digit year, `%m` month, `%d` day,
`%H` hour, `%M` minute, `%S` second. `"2015/06/01 00:00:00"` is
`"%Y/%m/%d %H:%M:%S"`. If the text has extra characters after the part you
want, cut it first: `col.str[:10]`.

---
<a id="C"></a>

## C — Missing values

pandas writes a missing value as `NaN` in a number column, `NaT` in a date
column, and `None` or `NaN` in a text column. `isna()` recognises all of them.

### C.1 Finding and counting

| You want | Write |
|---|---|
| is each value missing? | `col.isna()` |
| is each value present? | `col.notna()` |
| how many are missing | `col.isna().sum()` |
| what share is missing | `col.isna().mean()` |
| how many are present | `col.count()` |
| the rows where it is missing | `df[df["col"].isna()]` |
| rows with any missing value | `df[df.isna().any(axis=1)]` |

In [ ]:
print("missing:", plants["cover"].isna().sum(),
      " present:", plants["cover"].count(),
      " share missing:", plants["cover"].isna().mean())
plants[plants["cover"].isna()][["id", "forest", "cover"]]

**Never write `col == np.nan`.** It is `False` for every row, including the
missing ones, because a missing value is not equal to anything — not even to
another missing value. Use `isna()`.

In [ ]:
print((plants["cover"] == np.nan).sum(), "<- always 0")
print(plants["cover"].isna().sum(), "<- the real count")

### C.2 How missing values behave

| Operation | What a missing value does | Example |
|---|---|---|
| `sum`, `mean`, `min`, `max`, `median`, `std` | is **skipped** | `mean` divides by the number present |
| `count` | is **not counted** | `count() <= len()` |
| `size`, `len` | **is** counted | they count rows, not values |
| `+ - * /` with another column | the result is **missing** | `area * NaN` is `NaN` |
| `>`, `<`, `==` | the result is **`False`** | and so is the opposite comparison |
| `groupby(key)` | rows with a missing **key** are **dropped** | unless `dropna=False` |
| `value_counts()` | is **left out** | unless `dropna=False` |
| `nunique()` | is **not counted** as a value | unless `dropna=False` |
| `sort_values` | goes **last** | `na_position="first"` to change |

In [ ]:
cover = plants["cover"]
print("sum:", cover.sum(), " mean:", cover.mean(),
      " count:", cover.count(), " size:", cover.size)
print("cover > 15: ", (cover > 15).sum(),
      "  cover <= 15:", (cover <= 15).sum(),
      "  neither:", cover.isna().sum())

To make a sum *fail to skip* — missing if any value is missing — use
`col.sum(skipna=False)`. To get a missing result when a group has too few
values, `col.sum(min_count=1)` returns missing for a group with none at all
(plain `sum` returns 0 there).

### C.3 Removing

| You want | Write |
|---|---|
| drop rows with a blank in **any** column | `df.dropna()` |
| drop rows with a blank in **these** columns | `df.dropna(subset=["cover", "date"])` |
| drop rows where **every** column is blank | `df.dropna(how="all")` |
| keep rows with at least `n` values present | `df.dropna(thresh=n)` |
| drop **columns** that have blanks | `df.dropna(axis=1)` |

Plain `df.dropna()` is rarely what you want: one blank in one unimportant
column removes the row from every calculation. Name the columns.

In [ ]:
print("all rows:                ", len(plants))
print("dropna():                ", len(plants.dropna()))
print("dropna(subset=['cover']):", len(plants.dropna(subset=["cover"])))

### C.4 Filling

| A blank should become | Write |
|---|---|
| one fixed value | `col.fillna(0)` |
| a different value per column | `df.fillna({"cover": 0, "lifeform": "none"})` |
| the column's own mean or median | `col.fillna(col.median())` |
| the value from another column, row by row | `col.fillna(other_col)` |
| the mean or median of the row's **group** | `col.fillna(df.groupby("g")["col"].transform("median"))` |
| the previous row's value | `col.ffill()` — only for ordered data such as a time series |

`fillna` with a **column** fills each blank from the same row of that column,
which is what makes the group fill work: `transform` produces one value per
row.

In [ ]:
group_median = plants.groupby("forest")["cover"].transform("median")
pd.DataFrame({
    "forest": plants["forest"],
    "cover": plants["cover"],
    "fillna(0)": plants["cover"].fillna(0),
    "forest median": group_median,
    "fillna(forest median)": plants["cover"].fillna(group_median),
})

Charlie has no cover value at all, so it has no median, and its blanks stay
blank. That is correct. A fill can only use information that exists.

### C.5 Creating: turning a bad value into a missing one

| You want | Write |
|---|---|
| keep values where a condition holds, blank the rest | `col.where(condition)` |
| blank values where a condition holds | `col.mask(condition)` |
| blank specific values | `col.replace([-999, "N/A"], np.nan)` |
| blank what will not convert | `pd.to_numeric(col, errors="coerce")` |

In [ ]:
cover = plants["cover"]
pd.DataFrame({
    "cover": cover,
    "where(between 0 and 100)": cover.where(cover.between(0, 100)),
    "mask(cover > 100)": cover.mask(cover > 100),
})

In [ ]:
# A blank somebody typed a word into.  Normalise the text, then test.
words = plants["lifeform"].str.strip().str.lower()
pd.DataFrame({
    "lifeform": plants["lifeform"],
    "isna": plants["lifeform"].isna(),
    "is a missing word": words.isin(["unknown", ""]),
    "cleaned": plants["lifeform"].mask(words.isin(["unknown", ""])),
})

`where` and `mask` take an optional second argument, the value to put instead
of a blank: `col.where(col >= 0, 0)` replaces negatives with zero.

### C.6 Choosing a policy

| If a blank means | Then | And report |
|---|---|---|
| nothing happened (zero) | fill with 0 | that blanks were counted as zero |
| unknown | leave it; the summaries skip it | how many rows the number came from |
| not applicable | leave it | that those rows were never in scope |
| unknown, but similar rows are informative | fill from the group | how many values were filled, and from what |

Whatever you choose, **make the count visible**: put `count` next to every
`mean`.

In [ ]:
plants.groupby("forest").agg(
    records=("cover", "size"),
    with_cover=("cover", "count"),
    mean_cover=("cover", "mean"),
)

---
<a id="D"></a>

## D — Whole-column operations

A new column is one expression over existing columns. No loop.

### D.1 Arithmetic and comparison

| Kind | Operators | Notes |
|---|---|---|
| arithmetic | `+ - * / // % **` | between columns, or a column and one number |
| comparison | `== != < <= > >=` | gives a True/False column |
| and / or / not | `&  \|  ~` | **not** `and`, `or`, `not`; each comparison in its own parentheses |
| in a list | `col.isin([...])` | |
| in a range | `col.between(low, high)` | both ends included; `inclusive="left"` etc. to change |

In [ ]:
plants = make_plants()
plants["acres"] = plants["area"] * plants["percent"] / 100
plants["big_and_dense"] = (plants["area"] >= 4) & (plants["percent"] >= 50)
plants[["id", "area", "percent", "acres", "big_and_dense"]]

What a True/False column is good for:

| You want | Write |
|---|---|
| the rows where it is true | `df[flag]` |
| the rows where it is false | `df[~flag]` |
| how many are true | `flag.sum()` |
| what share is true | `flag.mean()` |
| is any / are all true | `flag.any()`, `flag.all()` |

### D.2 A value that depends on a condition

| Outcomes | Write |
|---|---|
| two | `np.where(condition, if_true, if_false)` |
| several, by conditions | `np.select([cond1, cond2], [value1, value2], default=other)` — first true condition wins |
| several, by numeric ranges | `pd.cut(col, bins=[...], labels=[...])` |
| several, with equally many rows in each | `pd.qcut(col, q=4, labels=[...])` |
| a lookup from a dictionary | `col.map({"thistle": "forb", "cheatgrass": "grass"})` |

In [ ]:
area = plants["area"]

plants["two"] = np.where(area >= 4, "large", "small")
plants["three"] = np.select([area < 1, area >= 100], ["small", "large"],
                            default="medium")
plants["cut"] = pd.cut(area, bins=[0, 1, 100, np.inf], right=False,
                       labels=["small", "medium", "large"])
plants[["id", "area", "two", "three", "cut"]]

**`pd.cut` and its edges.** `bins=[0, 1, 100, inf]` makes three ranges. With
`right=False` each range **includes its left edge** and excludes its right:
`[0, 1)`, `[1, 100)`, `[100, inf)`. The default, `right=True`, is the other
way round, and would also leave an area of exactly 0 outside every range.

**Missing values and conditions.** A comparison with a missing value is
`False`, so `np.where` and `np.select` send a missing input to the *false* or
*default* outcome — they give it a label it has not earned. `pd.cut` keeps it
missing. To keep it missing with `np.select`, blank the result afterwards:

In [ ]:
some_missing = pd.Series([0.5, np.nan, 250.0])
labels = np.select([some_missing < 1, some_missing >= 100], ["small", "large"],
                   default="medium")
print("np.select:          ", labels.tolist())
print("then .where(notna): ",
      pd.Series(labels).where(some_missing.notna()).tolist())
print("pd.cut:             ",
      pd.cut(some_missing, bins=[0, 1, 100, np.inf], right=False,
             labels=["small", "medium", "large"]).tolist())

### D.3 Other column helpers

| You want | Write |
|---|---|
| round | `col.round(2)` |
| limit to a range | `col.clip(lower=0, upper=100)` |
| absolute value | `col.abs()` |
| running total | `col.cumsum()` |
| change from the previous row | `col.diff()` |
| share of the column total | `col / col.sum()` |
| position if sorted | `col.rank(ascending=False)` |
| replace particular values | `col.replace({"Grass": "Graminoid"})` |

### D.4 When there really is no column operation

`col.apply(function)` calls a Python function once per value. It is a loop in
disguise: correct, and slow. Reach for it last, after looking through D.1–D.3
and section E. It is the right tool for a genuinely custom rule on a small
table, and the wrong one for arithmetic.

---
<a id="E"></a>

## E — Text and date columns

### E.1 Text: the `.str` methods

Every string method you know from core Python works on a whole column through
`.str`.

| You want | Write |
|---|---|
| lower / upper / title case | `col.str.lower()`, `.str.upper()`, `.str.title()` |
| remove surrounding spaces | `col.str.strip()` |
| length | `col.str.len()` |
| contains a piece of text | `col.str.contains("thist", na=False)` |
| starts / ends with | `col.str.startswith("kn")`, `.str.endswith("weed")` |
| replace text | `col.str.replace("$", "", regex=False)` |
| a slice of each value | `col.str[:10]` |
| split into pieces | `col.str.split(", ")` |
| one piece after splitting | `col.str.split(", ").str[0]` |
| look a key up in a dictionary cell | `col.str.get("key")` |
| join two text columns | `col_a + " " + col_b` |

In [ ]:
plants = make_plants()
pd.DataFrame({
    "lifeform": plants["lifeform"],
    "strip().lower()": plants["lifeform"].str.strip().str.lower(),
    "contains 'known'": plants["lifeform"].str.contains("known", na=False),
    "species[:3]": plants["species"].str[:3],
})

**`na=False`.** On a missing value, `.str.contains` returns missing, not
`False`, and a True/False column with a blank in it cannot be used to choose
rows. `na=False` says "a missing value does not contain it".

**`regex=False`.** `.str.replace` and `.str.contains` treat their pattern as a
regular expression by default in some pandas versions, where `$`, `.`, `(` and
`+` have special meanings. Pass `regex=False` when you mean the literal text.

### E.2 Dates: the `.dt` accessor

Works on a column that **is** a date (section B), not on text.

| You want | Write |
|---|---|
| year, month, day | `col.dt.year`, `.dt.month`, `.dt.day` |
| day of week, Monday = 0 | `col.dt.dayofweek` |
| name of day / month | `col.dt.day_name()`, `.dt.month_name()` |
| the date without the time | `col.dt.normalize()` |
| first day of the month | `col.dt.to_period("M").dt.start_time` |
| difference between two dates, in days | `(col_b - col_a).dt.days` |
| is it in a range | `col.between("2015-01-01", "2016-12-31")` |

In [ ]:
dates = pd.to_datetime(plants["date"], format="%Y/%m/%d", errors="coerce")
pd.DataFrame({
    "date": dates,
    "year": dates.dt.year.astype("Int64"),
    "month": dates.dt.month_name(),
    "weekday": dates.dt.day_name(),
    "valid year": dates.dt.year.between(1990, 2026),
})

The last column is how you remove impossible dates: `1900/01/01` is a real
date as far as pandas is concerned, so `errors="coerce"` leaves it alone.
`dates.where(dates.dt.year.between(1990, 2026))` blanks it. Note that the row
with no date is `False` too — a comparison with a missing value — which is
harmless here because it was already blank.

Time series tools — `resample`, `rolling`, `shift` — are in Session 7,
notebook 03.

---
<a id="F"></a>

## F — Split, apply, combine

`df.groupby(key)` splits. What you call on it applies and combines.

### F.1 The three kinds

| | Write | Result has | For |
|---|---|---|---|
| **aggregate** | `.agg(...)`, `.sum()`, `.mean()`, `.size()` | one row per group | summaries |
| **transform** | `.transform("mean")` | one row per original row | a group's value beside each row |
| **filter** | `.filter(lambda g: ...)` | some of the original rows | keeping whole groups |

In [ ]:
plants = make_plants()
plants["acres"] = plants["area"] * plants["percent"] / 100

groups = plants.groupby("forest")["acres"]
print("agg:      ", groups.sum().shape, "- one value per forest")
print("transform:", groups.transform("sum").shape, "- one value per row")
print("filter:   ", plants.groupby("forest").filter(lambda g: len(g) >= 3).shape,
      "- the rows of the forests that qualify")

### F.2 Aggregating

| You want | Write |
|---|---|
| one summary of one column | `df.groupby("g")["col"].sum()` |
| several summaries of one column | `df.groupby("g")["col"].agg(["count", "mean", "max"])` |
| several columns, each its own summary, named | `df.groupby("g").agg(new=("col", "how"), ...)` |
| rows per group | `df.groupby("g").size()` |
| several keys | `df.groupby(["g1", "g2"])...` |

**Summaries you can name as text:** `"size"`, `"count"`, `"nunique"`, `"sum"`,
`"mean"`, `"median"`, `"min"`, `"max"`, `"std"`, `"var"`, `"first"`, `"last"`.

| | counts | includes missing values? |
|---|---|---|
| `"size"` | rows in the group | yes |
| `"count"` | values present in the column | no |
| `"nunique"` | different values in the column | no |

In [ ]:
plants.groupby("forest", as_index=False).agg(
    records=("id", "size"),
    with_cover=("cover", "count"),
    species=("species", "nunique"),
    acres=("acres", "sum"),
    mean_cover=("cover", "mean"),
    largest=("area", "max"),
)

**A share within each group.** The mean of a True/False column is a share, and
that works per group too. Make the True/False column first:

In [ ]:
plants.assign(no_cover=plants["cover"].isna()).groupby("forest", as_index=False).agg(
    records=("id", "size"),
    no_cover_share=("no_cover", "mean"),
)

### F.3 Options of `groupby`

| Option | Default | What it does |
|---|---|---|
| `as_index=False` | `True` | keeps the key as an ordinary column instead of the index — usually what you want for a table you will go on using |
| `dropna=False` | `True` | keeps rows whose key is missing, as a group of their own |
| `sort=False` | `True` | keeps groups in order of first appearance instead of sorting the keys |

With the default `as_index=True`, `.reset_index()` afterwards does the same
job as `as_index=False`.

In [ ]:
cleaned = plants.assign(
    lifeform=plants["lifeform"].mask(
        plants["lifeform"].str.strip().str.lower().isin(["unknown", ""])))

print(cleaned.groupby("lifeform")["acres"].sum())
print()
print(cleaned.groupby("lifeform", dropna=False)["acres"].sum())
print()
print("the table's total:", cleaned["acres"].sum())

### F.4 Transforming

`transform` returns one value per original row, in the original order, so the
result can be assigned straight to a new column.

| You want, for every row | Write |
|---|---|
| its group's total | `df.groupby("g")["col"].transform("sum")` |
| its share of the group | `df["col"] / df.groupby("g")["col"].transform("sum")` |
| its difference from the group mean | `df["col"] - df.groupby("g")["col"].transform("mean")` |
| the size of its group | `df.groupby("g")["col"].transform("size")` |
| a blank filled from its group | `df["col"].fillna(df.groupby("g")["col"].transform("median"))` |
| its position within the group | `df.groupby("g").cumcount() + 1` |

In [ ]:
total = plants.groupby("forest")["acres"].transform("sum")
plants["share_of_forest"] = plants["acres"] / total
plants["forest_size"] = plants.groupby("forest")["id"].transform("size")
plants[["id", "forest", "acres", "share_of_forest", "forest_size"]]

**Dividing by a group total of zero** gives `NaN` (0 ÷ 0) or `inf` (something
÷ 0). If a zero total is possible, blank it first:
`df["col"] / total.where(total > 0)`.

### F.5 Filtering groups

Two ways to keep only the groups that meet a condition.

In [ ]:
# 1. filter: returns the ORIGINAL rows of the qualifying groups.
plants.groupby("forest").filter(lambda group: group["acres"].sum() > 10)[
    ["id", "forest", "acres"]]

In [ ]:
# 2. aggregate, then choose rows of the summary: returns one row per group.
summary = plants.groupby("forest", as_index=False).agg(acres=("acres", "sum"))
summary[summary["acres"] > 10]

### F.6 Sorting the result

A summary is a table; sort it like one. Give a second key so that ties come
out the same way every time:

```python
summary.sort_values(["acres", "forest"], ascending=[False, True])
```

Then `.head(n)` for the top `n`, and `.reset_index(drop=True)` for a clean
0, 1, 2, ... index.

### F.7 `groupby(...).apply`

`.apply(function)` hands each group to your function as a small table and
combines whatever comes back. It can do anything, slowly, and its output shape
depends on what the function returns. If `agg`, `transform` or `filter` can do
the job, use them.

---
<a id="G"></a>

## G — Reading and writing CSV and JSON

### G.1 CSV

| Option of `pd.read_csv` | Use it when |
|---|---|
| `usecols=["a", "b"]` | you need only some columns of a wide file |
| `nrows=1000` | you want a quick look at a big file |
| `dtype={"zip": "string"}` | a column of digits must stay text (leading zeros) |
| `na_values=["Unknown", "-999"]` | the file writes blanks as particular words or numbers |
| `parse_dates=["date"]` | a column is a date in a standard layout |
| `low_memory=False` | a large file gives a "mixed types" warning |
| `sep=";"` or `sep="\t"` | the separator is not a comma |
| `encoding="latin-1"` | the default UTF-8 fails on accented letters |

| Writing | |
|---|---|
| `df.to_csv(path, index=False)` | `index=False` almost always: otherwise the 0, 1, 2, ... index is written as an extra unnamed column |

### G.2 JSON

| The file looks like | Read | Write |
|---|---|---|
| `[{"a": 1, "b": 2}, {"a": 3, "b": 4}]` — a list of records | `pd.read_json(path)` | `df.to_json(path, orient="records")` |
| `{"a": 1, "b": 2}` on one line, the next record on the next line | `pd.read_json(path, lines=True)` | `df.to_json(path, orient="records", lines=True)` |

In [ ]:
small = plants[["id", "forest", "cover"]].head(3)

print(small.to_json(orient="records", indent=2))

In [ ]:
print(small.to_json(orient="records", lines=True))

A missing value is written as `null`. `indent=2` is for people; leave it out
for large files.

**Nested JSON.** When a record contains a dictionary — Yelp's `attributes` —
`read_json` puts the whole dictionary into one cell.

| You want | Write |
|---|---|
| one key out of each dictionary | `df["attributes"].str.get("WiFi")` |
| every key as its own column | `pd.json_normalize(df["attributes"].dropna().tolist())` |

In [ ]:
records = pd.DataFrame({
    "name": ["Cafe", "Diner", "Grill"],
    "attributes": [{"WiFi": "free", "Price": "2"}, {"WiFi": "no"}, None],
})
print(records["attributes"].str.get("WiFi").tolist())
pd.json_normalize(records["attributes"].dropna().tolist())

---
<a id="H"></a>

## H — Working style

### H.1 Return a new table; do not change the one you were given

A function that modifies its argument changes the caller's data behind their
back. Copy first, change the copy, return it.

In [ ]:
def add_acres(table):
    result = table.copy()
    result["acres"] = result["area"] * result["percent"] / 100
    return result


plants = make_plants()
with_acres = add_acres(plants)
print("acres" in plants.columns, "acres" in with_acres.columns)

`df.assign(new=...)` does the same in one expression: it returns a new table
with the column added and leaves `df` alone.

### H.2 `SettingWithCopyWarning`

You will see this warning when you choose some rows and then assign into the
result:

```python
alpha = plants[plants["forest"] == "Alpha"]
alpha["cover"] = 0          # SettingWithCopyWarning
```

pandas cannot tell whether `alpha` is its own table or a window onto `plants`,
so it cannot tell which one you meant to change. Say which:

| You meant | Write |
|---|---|
| a separate table to work on | `alpha = plants[plants["forest"] == "Alpha"].copy()` |
| to change those rows of `plants` itself | `plants.loc[plants["forest"] == "Alpha", "cover"] = 0` |

The related mistake is **chained assignment**, which often does nothing at
all:

```python
plants[plants["forest"] == "Alpha"]["cover"] = 0     # changes a temporary copy
plants.loc[plants["forest"] == "Alpha", "cover"] = 0 # changes plants
```

One `.loc[rows, column]`, never two pairs of square brackets in a row on the
left of `=`.

### H.3 Avoid `inplace=True`

Many methods accept `inplace=True`. It saves no memory, it returns `None`
(so `df = df.dropna(inplace=True)` destroys your table), and it prevents
chaining. Write `df = df.dropna(...)`.

### H.4 Method chaining

Several steps can be written as one expression, one step per line, inside
parentheses:

In [ ]:
(
    make_plants()
    .assign(acres=lambda t: t["area"] * t["percent"] / 100)
    .query("area > 0")
    .groupby("forest", as_index=False)
    .agg(records=("id", "size"), acres=("acres", "sum"))
    .sort_values("acres", ascending=False)
)

Inside `assign`, `lambda t: ...` receives the table **as it is at that step**.
That is how a later step can use a column an earlier step created.

Chaining reads well when each step is simple. When something goes wrong in the
middle, break the chain into named steps and look at each one.

### H.5 Two things that fail loudly, and should

| You wrote | The error | You meant |
|---|---|---|
| `if df["x"] > 3:` | *The truth value of a Series is ambiguous* | a column of True/False is not one True/False; use `.any()`, `.all()`, or choose rows with it |
| `(a > 1) and (b < 2)` on columns | the same | `&` in place of `and`, `\|` for `or`, `~` for `not` |

### H.6 After every step, check the shape

| After | Ask |
|---|---|
| choosing rows | did I keep roughly as many as I expected? |
| a `merge` | same number of rows as the left table? (Homework 6) |
| a `groupby` | one row per group — and do the groups' totals add up to the table's? |
| filling blanks | how many did I fill? how many are left? |

---
<a id="I"></a>

## I — From loop to pandas

The loop patterns from Sessions 3 and 4, and what each becomes.

| In core Python you wrote | In pandas |
|---|---|
| `total = 0` … `total += r["x"]` | `df["x"].sum()` |
| `count += 1` inside an `if` | `(df["x"] > 3).sum()` |
| `counts[key] = counts.get(key, 0) + 1` | `df["key"].value_counts()` |
| `totals[key] = totals.get(key, 0) + r["x"]` | `df.groupby("key")["x"].sum()` |
| a set of the values seen | `df["x"].unique()`, or `df["x"].nunique()` for how many |
| `[r for r in rows if r["x"] > 3]` | `df[df["x"] > 3]` |
| `[r["a"] * r["b"] for r in rows]` | `df["a"] * df["b"]` |
| `"big" if r["x"] > 3 else "small"` per row | `np.where(df["x"] > 3, "big", "small")` |
| `if`/`elif`/`else` per row | `np.select([...], [...], default=...)` |
| the largest value, and which record has it | `df["x"].max()`, `df.loc[df["x"].idxmax()]` |
| `sorted(rows, key=lambda r: (-r["x"], r["name"]))` | `df.sort_values(["x", "name"], ascending=[False, True])` |
| `if key in lookup:` for every row | `df["key"].isin(lookup)` |
| `lookup[r["key"]]` for every row | `df["key"].map(lookup)`, or a `merge` |
| `if r["x"] is None: continue` | summaries skip missing values; or `df.dropna(subset=["x"])` |
| `r["x"] if r["x"] is not None else 0` | `df["x"].fillna(0)` |
| `text.strip().lower()` per row | `df["text"].str.strip().str.lower()` |
| `json.loads(line)` for every line | `pd.read_json(path, lines=True)` |
| `csv.reader` and a loop | `pd.read_csv(path)` |
| write a header, then a row per record | `df.to_csv(path, index=False)` |

And the three that have no loop equivalent you would want to write:

| Question | pandas |
|---|---|
| each row as a share of its group's total | `df["x"] / df.groupby("g")["x"].transform("sum")` |
| each blank filled with its group's median | `df["x"].fillna(df.groupby("g")["x"].transform("median"))` |
| several summaries of several columns per group | `df.groupby("g").agg(a=("x", "sum"), b=("y", "mean"))` |

---

**Further reading**, in the pandas documentation:
[Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html),
[Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html),
[Working with text data](https://pandas.pydata.org/docs/user_guide/text.html), and
[IO tools](https://pandas.pydata.org/docs/user_guide/io.html).